# T08: Fabric Quickstart

## What You'll Learn

This is the **fastest path** to getting Spindle-generated data into a Microsoft Fabric
workspace. In this tutorial you will:

1. **Install** `sqllocks-spindle` inside a Fabric notebook
2. **Generate** a retail dataset
3. **Write** Parquet files to the attached Lakehouse
4. **Register** them as Delta tables using Spark
5. **Optionally** write to a Fabric SQL Database with `FabricSqlDatabaseWriter`
6. **Optionally** stream events with `FabricStreamWriter`

## Prerequisites

- A Microsoft Fabric workspace with a **Lakehouse** attached
- Running inside a **Fabric notebook** (Spark session available)
- For SQL Database steps: a Fabric SQL Database endpoint

## Time Estimate

**~10 minutes**

In [1]:
# Run this cell in your Fabric notebook to install Spindle.
# In Fabric, %pip is the recommended way to install packages.
%pip install sqllocks-spindle --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\suref\OneDrive\VSCode\AzureClients\forge-workspace\projects\fabric-datagen\.venv-win\Scripts\python.exe -m pip install --upgrade pip


## Step 1 — Generate Retail Data

**What we're about to do:** Create a `Spindle()` instance, then generate a retail dataset at `fabric_demo` scale.
This runs entirely in memory — no cluster resources needed.

**Why this matters:** Spindle generates pandas DataFrames, which are perfect for
local testing. In a Fabric notebook, you'd convert to Spark DataFrames only when writing to the Lakehouse.

In [2]:
from sqllocks_spindle import Spindle, RetailDomain

spindle = Spindle()
result = spindle.generate(domain=RetailDomain(), scale="fabric_demo", seed=42)

print("Generated Tables:")
for table_name, df in result.tables.items():
    print(f"  {table_name:<25} {len(df):>6,} rows")
print(f"\nTotal: {sum(len(df) for df in result.tables.values()):,} rows")

Generated Tables:
  customer                     200 rows
  address                      300 rows
  product_category              50 rows
  product                      100 rows
  promotion                    200 rows
  store                        150 rows
  order                      1,000 rows
  order_line                 2,500 rows
  return                       170 rows

Total: 4,670 rows


## Step 2 — Write Parquet to Lakehouse and Register as Delta Tables

**What we're about to do:** Convert each pandas DataFrame to a Spark DataFrame and
write it as a managed Delta table in the attached Lakehouse.

**Why this matters:** Once written as Delta tables, your data is immediately queryable
from the Lakehouse SQL analytics endpoint, Power BI DirectLake, and any other Fabric
workload. Delta format also gives you versioning, time travel, and ACID transactions.

> **Note:** This cell requires a Fabric notebook with an attached Lakehouse. If running
> locally, skip to the next section.

In [3]:
# --- Fabric Notebook Only ---
# Uncomment the block below when running in a Fabric notebook with an attached Lakehouse.

# TABLE_PREFIX = "spindle_retail_"
#
# for table_name, df in result.tables.items():
#     full_name = f"{TABLE_PREFIX}{table_name}"
#     (
#         spark.createDataFrame(df)
#             .write
#             .format("delta")
#             .mode("overwrite")
#             .option("overwriteSchema", "true")
#             .saveAsTable(full_name)
#     )
#     print(f"  {full_name:<40} {len(df):>7,} rows")
#
# print(f"\nDone — {len(result.tables)} tables written to Lakehouse as Delta.")

print("Uncomment the block above to write to your Fabric Lakehouse.")

Uncomment the block above to write to your Fabric Lakehouse.


## Step 3 (Optional) — Write to Fabric SQL Database

**What we're about to do:** Use `FabricSqlDatabaseWriter` to write the generated data
directly to a Fabric SQL Database endpoint using Managed Service Identity (MSI)
authentication — no connection strings or passwords needed.

**Why this matters:** Some workloads require SQL Database semantics (stored procedures,
views, indexed queries). `FabricSqlDatabaseWriter` handles table creation, schema
mapping, and batched inserts automatically.

> **Requires:** A Fabric SQL Database in your workspace and MSI-based access.

In [4]:
# --- Optional: Fabric SQL Database ---
# Uncomment to write to a Fabric SQL Database.

# from sqllocks_spindle.fabric import FabricSqlDatabaseWriter
#
# writer = FabricSqlDatabaseWriter(
#     auth="msi",                           # Uses Managed Service Identity
#     server="your-server.database.fabric.microsoft.com",
#     database="your-sql-database",
#     schema="spindle",
# )
#
# writer.write(result.tables)
# print("All tables written to Fabric SQL Database.")

print("Uncomment the block above to write to a Fabric SQL Database.")

Uncomment the block above to write to a Fabric SQL Database.


## Step 4 (Optional) — Stream Events with FabricStreamWriter

**What we're about to do:** Use `FabricStreamWriter` to send generated events to a
Fabric Eventstream or Event Hub for real-time ingestion scenarios.

**Why this matters:** If you're building a real-time analytics pipeline (KQL dashboards,
Eventhouse, or stream processing), you need a stream of events to test with.
`FabricStreamWriter` sends generated rows as events with configurable throughput.

> **Requires:** A Fabric Eventstream or Azure Event Hub connection string.

In [5]:
# --- Optional: Fabric Stream Writer ---
# Uncomment to stream events to an Eventstream or Event Hub.

# from sqllocks_spindle.fabric import FabricStreamWriter
#
# stream_writer = FabricStreamWriter(
#     connection_string="Endpoint=sb://...",   # Event Hub connection string
#     event_hub_name="spindle-events",
# )
#
# # Stream the "order" table as events (one event per row)
# stream_writer.send(result.tables["order"], batch_size=100)
# print("Order events streamed to Eventstream.")

print("Uncomment the block above to stream events to Fabric Eventstream.")

Uncomment the block above to stream events to Fabric Eventstream.


## What's Next?

You now have a complete path from Spindle generation to every major Fabric destination:
Lakehouse (Delta), SQL Database, and Eventstream.

| Notebook | What You'll Learn |
|----------|-------------------|
| **03: Fabric Lakehouse (reference)** | Advanced Lakehouse patterns — star schema, all 13 domains, Spark SQL queries |
| **T06: Star Schema Export** | Transform to dim/fact tables before writing to Lakehouse |
| **05: Streaming** | Advanced streaming patterns with IoT and Capital Markets data |

**Key takeaways:**
- `%pip install sqllocks-spindle` works directly in Fabric notebooks
- Spindle generates pandas DataFrames; convert to Spark only when writing
- Delta tables appear instantly in the Lakehouse SQL analytics endpoint
- `FabricSqlDatabaseWriter(auth="msi")` handles MSI auth automatically
- `FabricStreamWriter` enables real-time testing with synthetic events